# Chuyển đổi Motion CSV sang NPZ cho M2v6

Notebook này hướng dẫn cách chuyển đổi file CSV (từ retargeting) sang file NPZ
mà mjlab sử dụng khi huấn luyện.

## Quy trình
1. Đọc file CSV chứa góc khớp robot qua từng khung hình
2. Nội suy tốc độ khung hình (ví dụ: 30fps → 50fps)
3. Tính vận tốc tuyến tính và góc
4. Chạy forward kinematics trong MuJoCo để tính vị trí bộ phận cơ thể
5. Lưu tất cả vào file NPZ

## Yêu cầu
- Đã cài đặt mjlab (xem doc 02)
- Có file CSV từ bước retargeting

In [ ]:
import os
import numpy as np

# Thiết lập thư mục làm việc
MJLAB_DIR = "/home/nguyenld12/Documents/Humanoid_Tracking_Task/mjlab"
os.chdir(MJLAB_DIR)
print(f"Thư mục làm việc: {os.getcwd()}")

## 1. Cấu hình

Sửa các đường dẫn và tham số dưới đây cho phù hợp.

In [ ]:
# === SỬA CÁC THAM SỐ NÀY ===

# Đường dẫn file CSV đầu vào (từ retargeting)
INPUT_FILE = "/duong/dan/toi/file/chao_mung.csv"

# Đường dẫn file NPZ đầu ra
OUTPUT_FILE = "motions/chao_mung.npz"

# Tốc độ khung hình của file CSV
# - 30 nếu CSV từ GVHMR (30fps)
# - 120 nếu CSV từ motion capture chuyên nghiệp
INPUT_FPS = 30

# Tốc độ khung hình đầu ra
# PHẢI là 50 để khớp với tần số điều khiển của mjlab
OUTPUT_FPS = 50

# Có tạo video kiểm tra không
RENDER = True

# Chỉ xử lý một phần file CSV (None = toàn bộ)
# Ví dụ: LINE_RANGE = (100, 500) để xử lý dòng 100-500
LINE_RANGE = None

## 2. Kiểm tra File CSV Đầu vào

Xem trước nội dung file CSV để đảm bảo đúng định dạng.

In [ ]:
# Đọc và kiểm tra file CSV
assert os.path.exists(INPUT_FILE), f"Không tìm thấy file: {INPUT_FILE}"

data = np.loadtxt(INPUT_FILE, delimiter=",")
print(f"Kích thước dữ liệu: {data.shape}")
print(f"  - Số khung hình: {data.shape[0]}")
print(f"  - Số cột: {data.shape[1]}")
print(f"  - Thời lượng ước tính: {data.shape[0] / INPUT_FPS:.1f} giây")
print()

# Kiểm tra số cột
expected_cols = 3 + 4 + 27  # pos(3) + quat(4) + joints(27) = 34
if data.shape[1] == expected_cols:
    print(f"Đúng định dạng: {expected_cols} cột (3 pos + 4 quat + 27 joints)")
else:
    num_joints = data.shape[1] - 7
    print(f"Số cột: {data.shape[1]} (3 pos + 4 quat + {num_joints} joints)")
    print(f"Lưu ý: M2v6 cần 27 khớp, file này có {num_joints} khớp")

In [ ]:
# Xem trước dữ liệu
print("=== 5 dòng đầu tiên ===")
print()
print("Vị trí gốc (x, y, z):")
print(data[:5, :3])
print()
print("Quaternion gốc (qx, qy, qz, qw):")
print(data[:5, 3:7])
print()
print("6 khớp đầu tiên:")
print(data[:5, 7:13])
print()

# Kiểm tra chiều cao gốc hợp lý
z_mean = data[:, 2].mean()
z_min = data[:, 2].min()
z_max = data[:, 2].max()
print(f"Chiều cao gốc (z): trung bình={z_mean:.3f}m, min={z_min:.3f}m, max={z_max:.3f}m")
if z_mean < 0.5 or z_mean > 2.0:
    print("  CẢNH BÁO: Chiều cao gốc có vẻ bất thường. Robot M2v6 đứng cao ~1.1m.")

## 3. Chạy Chuyển đổi

Bước này sẽ:
- Nội suy khung hình (30fps → 50fps)
- Tính vận tốc
- Chạy forward kinematics trong MuJoCo
- Lưu file NPZ
- Tạo video kiểm tra (nếu RENDER = True)

In [ ]:
# Tạo thư mục đầu ra nếu chưa có
output_dir = os.path.dirname(OUTPUT_FILE)
if output_dir:
    os.makedirs(output_dir, exist_ok=True)

# Chạy script chuyển đổi
cmd_parts = [
    "MUJOCO_GL=egl",
    "uv run python -m mjlab.scripts.csv_to_npz_m2v6",
    f"--input-file {INPUT_FILE}",
    f"--output-file {OUTPUT_FILE}",
    f"--input-fps {INPUT_FPS}",
    f"--output-fps {OUTPUT_FPS}",
    f"--render {RENDER}",
]

if LINE_RANGE is not None:
    cmd_parts.append(f"--line-range '{LINE_RANGE}'")

cmd = " ".join(cmd_parts)
print(f"Đang chạy:\n{cmd}\n")
os.system(cmd)

## 4. Kiểm tra File NPZ Đầu ra

Xem nội dung file NPZ để đảm bảo chuyển đổi thành công.

In [ ]:
# Đọc và kiểm tra file NPZ
assert os.path.exists(OUTPUT_FILE), f"Không tìm thấy file NPZ: {OUTPUT_FILE}"

npz = np.load(OUTPUT_FILE)
print("=== Nội dung file NPZ ===")
print()
for key in npz.files:
    arr = npz[key]
    if isinstance(arr, np.ndarray):
        print(f"  {key:20s} | shape: {str(arr.shape):20s} | dtype: {arr.dtype}")
    else:
        print(f"  {key:20s} | value: {arr}")

print()
fps = npz['fps'][0]
num_frames = npz['joint_pos'].shape[0]
duration = num_frames / fps
print(f"FPS: {fps}")
print(f"Số khung hình: {num_frames}")
print(f"Thời lượng: {duration:.2f} giây")

In [ ]:
# Trực quan hóa dữ liệu
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
time = np.arange(num_frames) / fps

# Vị trí gốc
body_pos = npz['body_pos_w']
axes[0, 0].plot(time, body_pos[:, 0, 0], label='x')
axes[0, 0].plot(time, body_pos[:, 0, 1], label='y')
axes[0, 0].plot(time, body_pos[:, 0, 2], label='z')
axes[0, 0].set_title('Vị trí gốc (pelvis) theo thời gian')
axes[0, 0].set_xlabel('Thời gian (giây)')
axes[0, 0].set_ylabel('Vị trí (mét)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Góc khớp
joint_pos = npz['joint_pos']
for i in range(min(6, joint_pos.shape[1])):
    axes[0, 1].plot(time, joint_pos[:, i], alpha=0.7)
axes[0, 1].set_title('Góc 6 khớp đầu tiên theo thời gian')
axes[0, 1].set_xlabel('Thời gian (giây)')
axes[0, 1].set_ylabel('Góc (radian)')
axes[0, 1].grid(True, alpha=0.3)

# Vận tốc khớp
joint_vel = npz['joint_vel']
for i in range(min(6, joint_vel.shape[1])):
    axes[1, 0].plot(time, joint_vel[:, i], alpha=0.7)
axes[1, 0].set_title('Vận tốc 6 khớp đầu tiên')
axes[1, 0].set_xlabel('Thời gian (giây)')
axes[1, 0].set_ylabel('Vận tốc (rad/s)')
axes[1, 0].grid(True, alpha=0.3)

# Phân bố góc khớp
axes[1, 1].boxplot([joint_pos[:, i] for i in range(min(10, joint_pos.shape[1]))])
axes[1, 1].set_title('Phân bố góc 10 khớp đầu')
axes[1, 1].set_xlabel('Chỉ số khớp')
axes[1, 1].set_ylabel('Góc (radian)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = OUTPUT_FILE.replace('.npz', '_analysis.png')
plt.savefig(fig_path, dpi=100)
print(f"Đã lưu biểu đồ: {fig_path}")
plt.show()

## 5. Xem Video Kiểm tra

Nếu đã bật RENDER = True, một video MP4 sẽ được tạo cùng thư mục với file NPZ.
Video này cho thấy robot phát lại motion trong MuJoCo — giúp kiểm tra bằng mắt.

In [ ]:
# Xem video kiểm tra
from IPython.display import Video, display

video_file = OUTPUT_FILE.replace('.npz', '.mp4')
if os.path.exists(video_file):
    print(f"Video kiểm tra: {video_file}")
    display(Video(video_file, embed=True, width=640))
else:
    print(f"Không tìm thấy video: {video_file}")
    print("Đảm bảo RENDER = True khi chạy chuyển đổi.")

## 6. Checklist Trước khi Huấn luyện

Trước khi sang bước huấn luyện, kiểm tra:

- [ ] File NPZ đã được tạo thành công
- [ ] FPS đầu ra là 50
- [ ] Video kiểm tra cho thấy robot chuyển động hợp lý
- [ ] Chiều cao gốc hợp lý (~1.1m)
- [ ] Không có khung hình nào robot xuyên qua mặt đất
- [ ] Chuyển động mượt, không giật

## Bước tiếp theo

File NPZ đã sẵn sàng. Chuyển sang bước huấn luyện:
- [doc 07 - Huấn luyện Motion Imitation](../docs/07-huan-luyen-motion-imitation.md)
- Hoặc chạy script: `bash scripts/02_huan_luyen.sh`